# Database KPI Inventory

All ten tables in `data/port_analytics.db` — grain and KPI categories.

| Table | Grain | KPI category |
|---|---|---|
| `port_hourly_stats` | port × date × hour × vessel_type | Traffic volume, congestion |
| `vessel_visits` | MMSI × port × date | Dwell time, arrival/departure |
| `type_distribution` | port × date × vessel_type | Fleet composition |
| `port_navstatus_stats` | port × date × nav_status | Operational state |
| `port_speed_stats` | port × date × hour | Movement behaviour |
| `port_daily_flow` | port × date × vessel_type | Arrivals, departures, turnover |
| `aarhus_zone_hourly_stats` | zone × date × hour | Aarhus zone traffic |
| `aarhus_zone_visits` | MMSI × zone × date | Aarhus zone dwell |
| `aarhus_navstatus_stats` | zone × date × nav_status | Aarhus zone state |
| `aarhus_zone_speed_stats` | zone × date × hour | Aarhus approach speed |

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)

DB = Path('..') / 'data' / 'port_analytics.db'
conn = sqlite3.connect(DB)

def q(sql):
    return pd.read_sql_query(sql, conn)

print('Connected to:', DB.resolve())

Connected to: C:\Users\dell oem\Programmeren\Projecten\ais_project\data\port_analytics.db


## Table overview — row counts and columns

In [2]:
tables = [
    'port_hourly_stats', 'vessel_visits', 'type_distribution',
    'port_navstatus_stats', 'port_speed_stats', 'port_daily_flow',
    'aarhus_zone_hourly_stats', 'aarhus_zone_visits',
    'aarhus_navstatus_stats', 'aarhus_zone_speed_stats',
]
rows = []
for t in tables:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    cols = [r[1] for r in conn.execute(f'PRAGMA table_info({t})').fetchall()]
    rows.append({'table': t, 'rows': n, 'columns': ', '.join(cols)})
pd.DataFrame(rows).set_index('table')

,rows,columns
table,,
port_hourly_stats,19172,"port, date, hour, vessel_type, vessel_count"
vessel_visits,12686,"MMSI, port, date, first_seen, last_seen, vesse..."
type_distribution,745,"port, date, vessel_type, count"
port_navstatus_stats,750,"port, date, nav_status, ping_count"
port_speed_stats,3600,"port, date, hour, sog_mean, sog_median, sog_p9..."
port_daily_flow,910,"port, date, vessel_type, unique_vessels, entri..."
aarhus_zone_hourly_stats,2866,"zone, date, hour, vessel_count"
aarhus_zone_visits,2012,"MMSI, zone, date, first_seen, last_seen, vesse..."
aarhus_navstatus_stats,442,"zone, date, nav_status, ping_count"


---
## 1. Traffic Volume
### `port_hourly_stats` — unique vessel count per port × date × hour × vessel_type

> **Query pattern:** always `SUM(vessel_count) GROUP BY port, date, hour` before further aggregation.
> The vessel_type dimension means raw `AVG`/`MAX` operate on per-type counts, not totals.

In [3]:
q('SELECT * FROM port_hourly_stats ORDER BY port, date, hour, vessel_type LIMIT 10')

,port,date,hour,vessel_type,vessel_count
0,aalborg,2006-03-01,0,Cargo,38
1,aalborg,2006-03-01,0,Other,43
2,aalborg,2006-03-01,0,SAR,1
3,aalborg,2006-03-01,0,Tanker,14
4,aalborg,2006-03-01,0,Tug,2
5,aalborg,2006-03-01,1,Cargo,37
6,aalborg,2006-03-01,1,Other,7
7,aalborg,2006-03-01,1,SAR,1
8,aalborg,2006-03-01,1,Tanker,14
9,aalborg,2006-03-01,1,Tug,2


In [4]:
# KPI: daily traffic per port
q("""
    SELECT port, date, SUM(vessel_count) AS daily_vessels
    FROM port_hourly_stats
    GROUP BY port, date ORDER BY port, date LIMIT 15
""")

,port,date,daily_vessels
0,aalborg,2006-03-01,1592
1,aalborg,2006-04-01,1657
2,aalborg,2026-02-01,550
3,aalborg,2026-02-02,580
4,aalborg,2026-02-03,585
5,aalborg,2026-02-04,574
6,aalborg,2026-02-05,554
7,aalborg,2026-02-06,582
8,aalborg,2026-02-07,575
9,aalborg,2026-02-08,602


In [5]:
# KPI: peak hour per port per day
q("""
    WITH hourly_totals AS (
        SELECT port, date, hour, SUM(vessel_count) AS vessel_count
        FROM port_hourly_stats GROUP BY port, date, hour
    )
    SELECT h.port, h.date, h.hour, h.vessel_count AS peak_vessel_count
    FROM hourly_totals h
    WHERE (h.port, h.date, h.vessel_count) IN (
        SELECT port, date, MAX(vessel_count) FROM hourly_totals GROUP BY port, date
    )
    ORDER BY h.port, h.date LIMIT 15
""")

,port,date,hour,peak_vessel_count
0,aalborg,2006-03-01,0,98
1,aalborg,2006-04-01,0,101
2,aalborg,2026-02-01,23,25
3,aalborg,2026-02-02,0,26
4,aalborg,2026-02-02,4,26
5,aalborg,2026-02-02,8,26
6,aalborg,2026-02-02,11,26
7,aalborg,2026-02-03,0,26
8,aalborg,2026-02-03,10,26
9,aalborg,2026-02-04,8,26


In [6]:
# KPI: congestion heatmap — avg vessels by day-of-week × hour (0=Mon)
q("""
    WITH hourly_totals AS (
        SELECT port, date, hour, SUM(vessel_count) AS vessel_count
        FROM port_hourly_stats GROUP BY port, date, hour
    )
    SELECT port,
           (CAST(strftime('%w', date) AS INTEGER) + 6) % 7 AS dow,
           hour,
           ROUND(AVG(vessel_count), 2) AS avg_vessel_count
    FROM hourly_totals
    GROUP BY port, dow, hour ORDER BY port, dow, hour LIMIT 20
""")

,port,dow,hour,avg_vessel_count
0,aalborg,0,0,28.75
1,aalborg,0,1,25.75
2,aalborg,0,2,25.50
3,aalborg,0,3,25.25
4,aalborg,0,4,26.25
5,aalborg,0,5,27.50
6,aalborg,0,6,26.50
7,aalborg,0,7,26.00
8,aalborg,0,8,26.00
9,aalborg,0,9,25.00


### `type_distribution` — vessel type counts per port × date

In [7]:
# KPI: type mix per port with percentage
q("""
    SELECT port, vessel_type, SUM(count) AS total_vessels,
           ROUND(100.0 * SUM(count) / SUM(SUM(count)) OVER (PARTITION BY port), 1) AS pct
    FROM type_distribution
    GROUP BY port, vessel_type ORDER BY port, total_vessels DESC
""")

,port,vessel_type,total_vessels,pct
0,aalborg,Other,812,73.70
1,aalborg,Cargo,152,13.80
2,aalborg,Tanker,47,4.30
3,aalborg,Tug,39,3.50
4,aalborg,SAR,28,2.50
5,aalborg,Passenger,23,2.10
6,aalborg,Fishing,1,0.10
7,aarhus,Other,885,65.70
8,aarhus,Cargo,305,22.60
9,aarhus,Tanker,81,6.00


---
## 2. Port Utilisation — Dwell Time
### `vessel_visits` — one row per MMSI × port × date
Includes `entry_hour` / `exit_hour` for arrival pattern analysis.

In [8]:
q('SELECT * FROM vessel_visits LIMIT 8')

,MMSI,port,date,first_seen,last_seen,vessel_type,ping_count,dwell_minutes,entry_hour,exit_hour
0,0,aalborg,2006-04-01,2006-04-14 15:35:09,2006-04-16 16:21:41,Other,108,2926.50,15,16
1,0,copenhagen,2006-04-01,2006-04-08 23:58:45,2006-04-19 16:42:15,Other,606,15403.50,23,16
2,0,fredericia,2006-03-01,2006-03-14 10:04:18,2006-03-27 14:48:29,Other,139,19004.20,10,14
3,12345,aalborg,2006-03-01,2006-03-07 09:40:47,2006-03-31 06:25:43,Other,27,34364.90,9,6
4,12345,aalborg,2006-04-01,2006-04-26 10:51:13,2006-04-26 10:55:35,Other,26,4.40,10,10
5,14082,copenhagen,2006-03-01,2006-03-20 16:50:06,2006-03-20 16:50:06,Other,1,0.00,16,16
6,249130,fredericia,2006-03-01,2006-03-28 12:37:05,2006-03-29 09:45:25,Other,346,1268.30,12,9
7,1193046,aalborg,2006-03-01,2006-03-22 12:32:59,2006-03-23 11:25:32,Cargo,130,1372.60,12,11


In [9]:
# KPI: average dwell time by vessel type and port
q("""
    SELECT port, vessel_type, COUNT(*) AS visit_count,
           ROUND(AVG(dwell_minutes), 1) AS avg_dwell_min,
           ROUND(AVG(dwell_minutes) / 60.0, 2) AS avg_dwell_hrs
    FROM vessel_visits WHERE dwell_minutes > 0
    GROUP BY port, vessel_type ORDER BY port, avg_dwell_min DESC
""")

,port,vessel_type,visit_count,avg_dwell_min,avg_dwell_hrs
0,aalborg,Tanker,61,5275.50,87.92
1,aalborg,Cargo,190,4688.50,78.14
2,aalborg,SAR,35,2317.20,38.62
3,aalborg,Other,712,1778.80,29.65
4,aalborg,Passenger,57,1398.20,23.30
5,aalborg,Tug,39,1349.10,22.49
6,aalborg,Fishing,2,321.90,5.36
7,aarhus,Cargo,417,5825.00,97.08
8,aarhus,Tanker,93,5291.80,88.20
9,aarhus,Passenger,69,4919.80,82.00


In [10]:
# KPI: arrival hour patterns — what hour do vessels typically arrive?
q("""
    SELECT port, vessel_type, entry_hour, COUNT(*) AS arrivals
    FROM vessel_visits
    GROUP BY port, vessel_type, entry_hour
    ORDER BY port, vessel_type, entry_hour LIMIT 30
""")

,port,vessel_type,entry_hour,arrivals
0,aalborg,Cargo,0,48
1,aalborg,Cargo,1,4
2,aalborg,Cargo,2,2
3,aalborg,Cargo,3,3
4,aalborg,Cargo,4,6
5,aalborg,Cargo,5,1
6,aalborg,Cargo,6,12
7,aalborg,Cargo,7,8
8,aalborg,Cargo,8,15
9,aalborg,Cargo,9,8


---
## 3. Flow and Turnover
### `port_daily_flow` — entries, exits, unique vessels per port × date × vessel_type
Entry = MMSI seen today but NOT yesterday. Exit = seen today but NOT tomorrow.

In [11]:
q('SELECT * FROM port_daily_flow ORDER BY port, date, vessel_type LIMIT 10')

,port,date,vessel_type,unique_vessels,entries,exits,avg_dwell_minutes
0,aalborg,2006-03-01,Cargo,55,55,55,6531.60
1,aalborg,2006-03-01,Fishing,1,1,1,9.00
2,aalborg,2006-03-01,Other,11,11,11,14757.00
3,aalborg,2006-03-01,Passenger,1,1,1,57.30
4,aalborg,2006-03-01,SAR,1,1,1,26028.40
5,aalborg,2006-03-01,Tanker,22,22,22,7399.50
6,aalborg,2006-03-01,Tug,5,5,5,2812.60
7,aalborg,2006-04-01,Cargo,64,64,64,7426.20
8,aalborg,2006-04-01,Fishing,1,1,1,634.80
9,aalborg,2006-04-01,Other,17,17,17,12604.50


In [12]:
# KPI: average daily arrivals and departures
q("""
    SELECT port, vessel_type,
           ROUND(AVG(entries), 1)           AS avg_daily_entries,
           ROUND(AVG(exits), 1)             AS avg_daily_exits,
           ROUND(AVG(unique_vessels), 1)    AS avg_daily_vessels,
           ROUND(AVG(avg_dwell_minutes), 1) AS avg_dwell_min
    FROM port_daily_flow
    GROUP BY port, vessel_type ORDER BY port, avg_daily_vessels DESC
""")

,port,vessel_type,avg_daily_entries,avg_daily_exits,avg_daily_vessels,avg_dwell_min
0,aalborg,Other,3.10,3.20,23.90,2127.50
1,aalborg,Cargo,5.00,5.00,6.40,1299.70
2,aalborg,Tanker,2.60,2.60,3.20,1379.60
3,aalborg,Passenger,0.10,0.10,2.00,1375.10
4,aalborg,Tug,0.90,0.90,1.70,782.30
5,aalborg,SAR,0.90,0.80,1.60,2106.30
6,aalborg,Fishing,1.00,1.00,1.00,321.90
7,aarhus,Other,3.70,3.70,18.10,2367.80
8,aarhus,Cargo,9.70,9.70,13.90,1655.90
9,aarhus,Tug,0.90,0.90,4.20,2459.40


In [13]:
# KPI: fleet turnover rate = entries / unique_vessels
q("""
    SELECT port, vessel_type,
           ROUND(AVG(CAST(entries AS REAL) / NULLIF(unique_vessels, 0)), 2) AS turnover_rate
    FROM port_daily_flow
    GROUP BY port, vessel_type ORDER BY port, turnover_rate DESC
""")

,port,vessel_type,turnover_rate
0,aalborg,Fishing,1.00
1,aalborg,Tanker,0.58
2,aalborg,SAR,0.52
3,aalborg,Cargo,0.48
4,aalborg,Tug,0.39
5,aalborg,Other,0.16
6,aalborg,Passenger,0.07
7,aarhus,SAR,1.00
8,aarhus,Tanker,0.55
9,aarhus,Cargo,0.48


---
## 4. Movement Behaviour
### `port_speed_stats` — SOG distribution per port × date × hour
`pct_stationary` = % of pings with SOG < 0.5 kn (proxy for stopped / waiting).

In [14]:
q('SELECT * FROM port_speed_stats ORDER BY port, date, hour LIMIT 10')

,port,date,hour,sog_mean,sog_median,sog_p95,pct_stationary
0,aalborg,2006-03-01,0,0.06,0.00,0.10,99.20
1,aalborg,2006-03-01,1,0.09,0.00,0.10,98.80
2,aalborg,2006-03-01,2,0.07,0.00,0.10,99.00
3,aalborg,2006-03-01,3,0.09,0.00,0.10,98.50
4,aalborg,2006-03-01,4,0.14,0.00,0.10,97.00
5,aalborg,2006-03-01,5,0.92,0.00,8.90,85.60
6,aalborg,2006-03-01,6,0.13,0.00,0.10,97.80
7,aalborg,2006-03-01,7,0.18,0.00,0.10,96.90
8,aalborg,2006-03-01,8,0.54,0.00,4.52,91.30
9,aalborg,2006-03-01,9,0.54,0.00,3.60,93.40


In [15]:
# KPI: movement behaviour summary per port
q("""
    SELECT port,
           ROUND(AVG(sog_mean), 2)       AS avg_sog_mean,
           ROUND(AVG(sog_median), 2)     AS avg_sog_median,
           ROUND(AVG(sog_p95), 2)        AS avg_sog_p95,
           ROUND(AVG(pct_stationary), 1) AS avg_pct_stationary
    FROM port_speed_stats GROUP BY port ORDER BY avg_pct_stationary DESC
""")

,port,avg_sog_mean,avg_sog_median,avg_sog_p95,avg_pct_stationary
0,aalborg,1.32,0.04,7.02,90.10
1,esbjerg,0.71,0.00,5.12,88.60
2,fredericia,1.46,0.30,6.61,82.50
3,aarhus,1.73,0.10,10.97,79.90
4,copenhagen,2.27,0.05,11.70,73.70


In [16]:
# KPI: intraday SOG pattern
q("""
    SELECT port, hour,
           ROUND(AVG(sog_mean), 2)       AS avg_sog,
           ROUND(AVG(pct_stationary), 1) AS avg_pct_stationary
    FROM port_speed_stats GROUP BY port, hour ORDER BY port, hour LIMIT 30
""")

,port,hour,avg_sog,avg_pct_stationary
0,aalborg,0,0.13,94.60
1,aalborg,1,0.16,97.40
2,aalborg,2,0.13,97.60
3,aalborg,3,0.22,95.60
4,aalborg,4,0.23,95.10
5,aalborg,5,0.32,93.50
6,aalborg,6,0.36,90.00
7,aalborg,7,1.23,85.00
8,aalborg,8,0.77,80.00
9,aalborg,9,1.77,81.70


---
## 5. Operational State
### `port_navstatus_stats` — ping count by navigational status per port × date

In [17]:
q('SELECT * FROM port_navstatus_stats ORDER BY port, date, ping_count DESC LIMIT 10')

,port,date,nav_status,ping_count
0,aalborg,2006-03-01,Under way using engine,39218
1,aalborg,2006-03-01,Moored,29770
2,aalborg,2006-03-01,Not under command,5208
3,aalborg,2006-03-01,Unknown value,4950
4,aalborg,2006-03-01,Under way sailing,897
5,aalborg,2006-03-01,Reserved for future use [12],770
6,aalborg,2006-03-01,Restricted maneuverability,452
7,aalborg,2006-03-01,At anchor,329
8,aalborg,2006-03-01,Engaged in fishing,5
9,aalborg,2006-04-01,Under way using engine,265763


In [18]:
# KPI: navigational status breakdown per port
q("""
    WITH totals AS (
        SELECT port, SUM(ping_count) AS port_total FROM port_navstatus_stats GROUP BY port
    )
    SELECT n.port, n.nav_status,
           SUM(n.ping_count) AS ping_count,
           ROUND(100.0 * SUM(n.ping_count) / t.port_total, 1) AS pct
    FROM port_navstatus_stats n JOIN totals t ON n.port = t.port
    GROUP BY n.port, n.nav_status ORDER BY n.port, ping_count DESC
""")

,port,nav_status,ping_count,pct
0,aalborg,Unknown value,799432,49.50
1,aalborg,Under way using engine,425266,26.30
2,aalborg,Moored,342253,21.20
3,aalborg,Restricted maneuverability,29936,1.90
4,aalborg,Under way sailing,8526,0.50
5,aalborg,Not under command,5208,0.30
6,aalborg,Engaged in fishing,4030,0.20
7,aalborg,Reserved for future use [12],770,0.00
8,aalborg,At anchor,669,0.00
9,aalborg,Aground,2,0.00


---
## 6. Aarhus Sub-Zone Analysis
Zones: `outer_approach` | `anchorage` | `south_terminal` | `north_terminal`

In [19]:
# Zone traffic overview
q("""
    SELECT zone, SUM(vessel_count) AS total_vessel_hours,
           ROUND(AVG(vessel_count), 2) AS avg_per_hour
    FROM aarhus_zone_hourly_stats GROUP BY zone ORDER BY total_vessel_hours DESC
""")

,zone,total_vessel_hours,avg_per_hour
0,anchorage,11006,15.29
1,south_terminal,10285,14.28
2,outer_approach,2273,3.16
3,north_terminal,2194,3.11


In [20]:
# KPI: dwell time per zone and vessel type
q("""
    SELECT zone, vessel_type, COUNT(*) AS visits,
           ROUND(AVG(dwell_minutes), 1) AS avg_dwell_min,
           ROUND(AVG(dwell_minutes) / 60.0, 2) AS avg_dwell_hrs
    FROM aarhus_zone_visits WHERE dwell_minutes > 0
    GROUP BY zone, vessel_type ORDER BY zone, avg_dwell_min DESC
""")

,zone,vessel_type,visits,avg_dwell_min,avg_dwell_hrs
0,anchorage,Cargo,141,9226.20,153.77
1,anchorage,HSC,9,8172.20,136.20
2,anchorage,Passenger,65,5185.70,86.43
3,anchorage,Tanker,70,4932.70,82.21
4,anchorage,Other,211,4309.10,71.82
5,anchorage,Tug,113,2838.20,47.30
6,anchorage,SAR,2,0.90,0.01
7,north_terminal,Passenger,4,2915.60,48.59
8,north_terminal,Cargo,14,1837.80,30.63
9,north_terminal,Other,137,1629.20,27.15


In [21]:
# KPI: anchorage nav-status — at-anchor % is a waiting-queue proxy
q("""
    WITH totals AS (
        SELECT zone, SUM(ping_count) AS zone_total FROM aarhus_navstatus_stats GROUP BY zone
    )
    SELECT n.zone, n.nav_status,
           SUM(n.ping_count) AS pings,
           ROUND(100.0 * SUM(n.ping_count) / t.zone_total, 1) AS pct
    FROM aarhus_navstatus_stats n JOIN totals t ON n.zone = t.zone
    GROUP BY n.zone, n.nav_status ORDER BY n.zone, pings DESC
""")

,zone,nav_status,pings,pct
0,anchorage,Under way using engine,1419121,68.70
1,anchorage,Moored,315300,15.30
2,anchorage,Unknown value,191605,9.30
3,anchorage,Reserved for future amendment [HSC],101506,4.90
4,anchorage,Reserved for future use [11],31992,1.50
5,anchorage,Under way sailing,3108,0.20
6,anchorage,Restricted maneuverability,1017,0.00
7,anchorage,At anchor,546,0.00
8,anchorage,Not under command,310,0.00
9,north_terminal,Unknown value,117112,94.40


In [22]:
# KPI: outer_approach speed profile by hour
q("""
    SELECT zone, hour,
           ROUND(AVG(sog_mean), 2)   AS avg_sog,
           ROUND(AVG(sog_median), 2) AS med_sog,
           ROUND(AVG(sog_p95), 2)    AS p95_sog
    FROM aarhus_zone_speed_stats WHERE zone = 'outer_approach'
    GROUP BY zone, hour ORDER BY hour
""")

,zone,hour,avg_sog,med_sog,p95_sog
0,outer_approach,0,0.31,0.24,0.79
1,outer_approach,1,0.94,0.83,2.14
2,outer_approach,2,0.80,0.84,1.75
3,outer_approach,3,1.52,1.89,2.69
4,outer_approach,4,1.32,1.37,3.11
5,outer_approach,5,2.31,2.07,6.34
6,outer_approach,6,4.88,5.80,6.97
7,outer_approach,7,2.08,2.16,4.39
8,outer_approach,8,4.99,6.33,7.96
9,outer_approach,9,4.65,5.73,7.63


---
## Summary — KPI to table mapping

| KPI | Primary table | Key columns |
|---|---|---|
| Daily vessel count | `port_hourly_stats` | SUM(vessel_count) GROUP BY port, date |
| Peak congestion hour | `port_hourly_stats` | CTE then MAX(vessel_count) per port, date |
| Congestion heatmap (dow × hour) | `port_hourly_stats` | CTE then AVG(vessel_count) per dow, hour |
| Type-stratified congestion | `port_hourly_stats` | GROUP BY vessel_type before AVG |
| Fleet composition (type %) | `type_distribution` | SUM(count) with window pct per port |
| Average dwell time | `vessel_visits` | AVG(dwell_minutes) by port, vessel_type |
| Arrival hour patterns | `vessel_visits` | entry_hour distribution |
| Daily arrivals / departures | `port_daily_flow` | entries, exits columns |
| Fleet turnover rate | `port_daily_flow` | entries / unique_vessels |
| Stationary % (waiting proxy) | `port_speed_stats` | pct_stationary |
| Average speed by port/hour | `port_speed_stats` | sog_mean, sog_p95 |
| Moored / underway / at anchor % | `port_navstatus_stats` | ping_count with window pct |
| Aarhus zone traffic | `aarhus_zone_hourly_stats` | vessel_count by zone, hour |
| Aarhus zone dwell / turnaround | `aarhus_zone_visits` | dwell_minutes by zone, vessel_type |
| Aarhus waiting queue | `aarhus_navstatus_stats` | At anchor % in anchorage zone |
| Aarhus approach speed | `aarhus_zone_speed_stats` | sog_mean in outer_approach |

In [23]:
conn.close()